# Script Outline

1) what is the trend of Drive Alone to commute from Year 2018-2022?
2) what is the trend of Work From Home from Year 2018-2022?

- Prepare Workspace
- Import Data
- Data Visualization

## Prepare Workspace

#### Import Packages

In [ ]:
# General
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import math


# Geographic
import geopandas as gpd
from census import Census
from us import states
import censusdata as acs


# Plotting
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

# Chart Studio
import chart_studio
chart_studio.tools.set_credentials_file(username='jfontes94', api_key='EaE848TXZBs2iiTsNIvu')
import chart_studio.plotly as py

#### File paths

In [ ]:
# Working directory
path_projects = os.path.dirname(os.path.dirname(os.getcwd()))
print(path_projects)

# Set file paths
rootpath = os.path.join(path_projects, 'SACOG Exam' )
path_in     = os.path.join(rootpath, 'Raw Data'     )
path_out    = os.path.join(rootpath, 'Python Output')
path_config = os.path.join(rootpath, 'config'       )
path_shiny  = os.path.join(rootpath, 'shiny', 'inputs')

## Import Data

In [ ]:
df_acs1 = pd.read_excel(os.path.join(path_out, 'Step 02a_ACS1 Transportation Metrics_Counties_2024-02-24.xlsx'))
df_acs5_counties = pd.read_excel(os.path.join(path_out, 'Step 02b_ACS5 Transportation Metrics_Counties_2024-02-24.xlsx'))
df_acs5_tracts   = pd.read_excel(os.path.join(path_out, 'Step 02b_ACS5 Transportation Metrics_Tracts_2024-02-24.xlsx'))

In [ ]:
print(df_acs1.shape)
df_acs1.head(6)

In [ ]:
print(df_acs5_counties.shape)
df_acs5_counties.head(3)

In [ ]:
print(df_acs5_tracts.shape)
df_acs5_tracts.head(3)

## Data Visualization

In [ ]:
# Plot settings
plot_template = 'ggplot2'
counties_acs1 = ['El Dorado', 'Placer', 'Sacramento', 'Yolo']
counties_acs5 = ['El Dorado', 'Placer', 'Sacramento', 'Yolo', 'Sutter', 'Yuba']

### ACS 1 Year Estimates

In [ ]:
# Subset data for plotting
df_plot1 = df_acs1.rename(columns = {'County Name':'County'})
df_plot1.head()

#### Tables

In [ ]:
# Organize the cleaned ACS data in nice table format by metric for exporting

metrics = ["Population"
          , "Population commuting alone"  
          , "Population working from home"
          , "Population commuting alone (%)"  
          , "Population working from home (%)"
          , "Population commuting alone_per 1k pop"  
          , "Population working from home_per 1k pop"
          , "Population commuting alone_per 1k acres"  
          , "Population working from home_per 1k acres"]

df_exports = []

for metric in tqdm(metrics):
    df_plot = df_plot1[['County', 'Year', metric]]
    df_plot = df_plot.pivot(index = 'County', columns = 'Year', values = metric)
    df_plot.reset_index(inplace = True)
    df_exports.append(df_plot)

In [ ]:
# # Export the cleaned ACS data in nice table format by metric

# metrics_abrv = ["Pop"
#           , "Pop commute alone"  
#           , "Pop WFH"
#           , "Pop commute alone pct"  
#           , "Pop WFH pct"
#           , "Pop commute alone_norm pop"  
#           , "Pop WFH_norm pop"
#           , "Pop commute alone_norm acres"  
#           , "Pop WFH_norm acres"]


# with pd.ExcelWriter(os.path.join(path_out, 'tables', 'Annual Estimates by Year Counties ACS1.xlsx')) as writer:
#     df_exports[0].to_excel(writer, sheet_name = metrics_abrv[0], index=False)
#     df_exports[1].to_excel(writer, sheet_name = metrics_abrv[1], index=False)
#     df_exports[2].to_excel(writer, sheet_name = metrics_abrv[2], index=False)
#     df_exports[3].to_excel(writer, sheet_name = metrics_abrv[3], index=False)
#     df_exports[4].to_excel(writer, sheet_name = metrics_abrv[4], index=False)
#     df_exports[5].to_excel(writer, sheet_name = metrics_abrv[5], index=False)
#     df_exports[6].to_excel(writer, sheet_name = metrics_abrv[6], index=False)
#     df_exports[7].to_excel(writer, sheet_name = metrics_abrv[7], index=False)
#     df_exports[8].to_excel(writer, sheet_name = metrics_abrv[8], index=False)

#### Bar Charts

In [ ]:
# Reformat the data for plotting
df_plot2 = df_plot1.copy()
df_plot2 = df_plot2.melt(id_vars = ['COUNTYFP', 'Year', 'County'], value_name = 'Value', var_name = 'Metric')
df_plot2.head()

In [ ]:
# Subset the data for specific bar plot
# Make bar plots
# Export to Chart Studio to put in dashboard

df_plot = df_plot2[df_plot2['Metric'].isin(["Population commuting alone", "Population working from home"])]
df_plot = df_plot[df_plot['County'].isin(["Sacramento", "Placer", "El Dorado"])]


fig = px.bar(df_plot
             , x = "County"
             , y = "Value"
             , color = "Metric"
             , title = "Total Population Commuting vs Working Frome Home"
             , template = plot_template
             , facet_col = 'Year'
             , labels={'County': '', 'Value':'Population'}
            )

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))


fig.show()
# py.plot(fig, filename = "".join(["Total Population Commuting vs WFH", "_ACS1", '_BAR']))

In [ ]:
# Subset the data for specific bar plot
# Make bar plots
# Export to Chart Studio to put in dashboard

df_plot = df_plot2[df_plot2['Metric'].isin(["Population commuting alone (%)", "Population working from home (%)"])]
df_plot = df_plot[df_plot['County'].isin(["Sacramento", "Placer", "El Dorado"])]

fig = px.bar(df_plot
             , x = "County"
             , y = "Value"
             , color = "Metric"
             , title = "Total Population Commuting vs Working Frome Home (%)"
             , template = plot_template
             , facet_col = 'Year'
             , labels={'County': '', 'Value':'Population'}
            )

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))


fig.show()
# py.plot(fig, filename = "".join(["Total Population Commuting vs WFH (%)", "_ACS1", '_BAR']))

#### Line Graphs

In [ ]:
# Subset the data for specific line graph
# Make line graphs
# Export to Chart Studio to put in dashboard

## Chart 1 --
# colors = px.colors.qualitative.Plotly

# df_plot = df_plot1.copy()


# fig = go.Figure()

# for i,r in enumerate(df_plot['County'].unique()):
#     dff = df_plot.query('County == @r')
#     fig.add_traces(go.Scatter(
#         x = dff['Year']
#          , y = dff['Population commuting alone']
#          , mode = "lines+markers"
#          , name = str(r)
#          , legendgroup = 'Population commuting alone'
#          , legendgrouptitle = dict(text = 'Commuting alone')
#          , line = dict(color = colors[i])
#        )
#     )
    
#     fig.add_traces(go.Scatter(
#         x = dff['Year']
#          , y = dff['Population working from home']
#          , mode = "lines+markers"
#          , name = str(r)
#          , legendgroup = 'Population working from home'
#          , legendgrouptitle = dict(text = 'Working from home')
#          , line = dict(color = colors[i], dash = 'dash')
#        )
#     )



## Chart 2 --

df_plot = df_plot2[df_plot2['Metric'].isin(["Population commuting alone", "Population working from home"])]

fig = px.line(df_plot
                 , x = "Year"
                 , y = 'Value'
                 , color = "County"
                 , line_dash = 'Metric'
                 , markers = True
                )
    
fig.update_layout(title = 'Total Population')

    
# py.plot(fig, filename = "".join(['Population Commuting Alone vs WFH', '_ACS1', '_LINES']))
fig.show()

In [ ]:
# Subset the data for specific line graph
# Make line graphs
# Export to Chart Studio to put in dashboard


## Chart 1 --
# colors = px.colors.qualitative.Plotly

# df_plot = df_plot1.copy()


# fig = go.Figure()

# for i,r in enumerate(df_plot['County'].unique()):
#     dff = df_plot.query('County == @r')
#     fig.add_traces(go.Scatter(
#         x = dff['Year']
#          , y = dff['Population commuting alone (%)']
#          , mode = "lines+markers"
#          , name = str(r)
#          , legendgroup = 'Population commuting alone (%)'
#          , legendgrouptitle = dict(text = 'Commuting alone (%)')
#          , line = dict(color = colors[i])
#        )
#     )
    
#     fig.add_traces(go.Scatter(
#         x = dff['Year']
#          , y = dff['Population working from home (%)']
#          , mode = "lines+markers"
#          , name = str(r)
#          , legendgroup = 'Population working from home (%)'
#          , legendgrouptitle = dict(text = 'Working from home (%)')
#          , line = dict(color = colors[i], dash = 'dash')
#        )
#     )



## Chart 2 --
df_plot = df_plot2[df_plot2['Metric'].isin(["Population commuting alone (%)", "Population working from home (%)"])]

fig = px.line(df_plot
                 , x = "Year"
                 , y = 'Value'
                 , color = "County"
                 , line_dash = 'Metric'
                 , markers = True
                )

fig.update_layout(title = 'Percent of Total Population')


# py.plot(fig, filename = "".join(['Population Commuting Alone vs WFH (%)', '_ACS1', '_LINES']))
fig.show()

In [ ]:
# Subset the data for specific line graph
# Make line graphs
# Export to Chart Studio to put in dashboard


## Chart 1 --
# colors = px.colors.qualitative.Plotly

# df_plot = df_plot1.copy()

# fig = go.Figure()

# for i,r in enumerate(df_plot['County'].unique()):
#     dff = df_plot.query('County == @r')
#     fig.add_traces(go.Scatter(
#         x = dff['Year']
#          , y = dff['Population commuting alone_per 1k acres']
#          , mode = "lines+markers"
#          , name = str(r)
#          , legendgroup = 'Population commuting alone_per 1k acres'
#          , legendgrouptitle = dict(text = 'Commuting alone per 1k acres')
#          , line = dict(color = colors[i])
#        )
#     )
    
#     fig.add_traces(go.Scatter(
#         x = dff['Year']
#          , y = dff['Population working from home_per 1k acres']
#          , mode = "lines+markers"
#          , name = str(r)
#          , legendgroup = 'Population working from home_per 1k acres'
#          , legendgrouptitle = dict(text = 'Working from home per 1k acres')
#          , line = dict(color = colors[i], dash = 'dash')
#        )
#     )
    
    
    

## Chart 2 --
df_plot = df_plot2[df_plot2['Metric'].isin(["Population commuting alone_per 1k acres", "Population working from home_per 1k acres"])]

fig = px.line(df_plot
                 , x = "Year"
                 , y = 'Value'
                 , color = "County"
                 , line_dash = 'Metric'
                 , markers = True
                )

fig.update_layout(title = 'Total Population Per 1,000 Acres')


# py.plot(fig, filename = "".join(['Population Commuting Alone vs WFH Per 1k Acres', '_ACS1', '_LINES']))
fig.show()

### ACS 5-Year Estimates

#### Counties

In [ ]:
# # Subset data for plotting
# df_plot5_counties = df_acs5_counties.copy()
# df_plot5_counties = df_plot5_counties.merge(df_plot1[['COUNTYFP', 'County Name']].drop_duplicates(), on = 'COUNTYFP')
# df_plot5_counties.head()

#### Tables

In [ ]:
# metrics = ["Population"
#           , "Population commuting alone"  
#           , "Population working from home"
#           , "Population commuting alone (%)"  
#           , "Population working from home (%)"
#           , "Population commuting alone_per 1k pop"  
#           , "Population working from home_per 1k pop"
#           , "Population commuting alone_per 1k acres"  
#           , "Population working from home_per 1k acres"]

# df_exports = []

# for metric in tqdm(metrics):
#     df_plot = df_plot5_counties[['County Name', 'Year', metric]]
#     df_plot = df_plot.pivot(index = 'County Name', columns = 'Year', values = metric)
#     df_plot.reset_index(inplace = True)
#     df_exports.append(df_plot)


In [ ]:
# metrics_abrv = ["Pop"
#           , "Pop commute alone"  
#           , "Pop WFH"
#           , "Pop commute alone pct"  
#           , "Pop WFH pct"
#           , "Pop commute alone_norm pop"  
#           , "Pop WFH_norm pop"
#           , "Pop commute alone_norm acres"  
#           , "Pop WFH_norm acres"]


# with pd.ExcelWriter(os.path.join(path_out, 'tables', 'Annual Estimates by Year Counties ACS5 RAW.xlsx')) as writer:
#     df_exports[0].to_excel(writer, sheet_name = metrics_abrv[0], index=False)
#     df_exports[1].to_excel(writer, sheet_name = metrics_abrv[1], index=False)
#     df_exports[2].to_excel(writer, sheet_name = metrics_abrv[2], index=False)
#     df_exports[3].to_excel(writer, sheet_name = metrics_abrv[3], index=False)
#     df_exports[4].to_excel(writer, sheet_name = metrics_abrv[4], index=False)
#     df_exports[5].to_excel(writer, sheet_name = metrics_abrv[5], index=False)
#     df_exports[6].to_excel(writer, sheet_name = metrics_abrv[6], index=False)
#     df_exports[7].to_excel(writer, sheet_name = metrics_abrv[7], index=False)
#     df_exports[8].to_excel(writer, sheet_name = metrics_abrv[8], index=False)



#### Plots

In [ ]:
# # Export to html
# metrics = ["Population"
#           , "Population commuting alone"  
#           , "Population working from home"
#           , "Population commuting alone (%)"  
#           , "Population working from home (%)"
#           , "Population commuting alone_per 1k pop"  
#           , "Population working from home_per 1k pop"
#           , "Population commuting alone_per 1k acres"  
#           , "Population working from home_per 1k acres"]

# # Loop bar plots to html
# for metric in tqdm(metrics):
#     fig = px.bar(df_plot5_counties
#                  , x = "Year"
#                  , y = metric
#                  , color = "County Name"
#                  , title = metric
#                  , template = plot_template
#                  , barmode = 'group'
#                 )
#     fig.write_html(
#         os.path.join(
#             path_out
#             , 'plotly'
#             , 'ACS5'
#             , 'bars'
#             , ''.join([metric
#                        , '_bar_'
#                        , '_ACS5_'
#                        , '_Counties_'
#                        , date.today().strftime("%Y-%m-%d")
#                        , '.html'])
#         )
#     )
    
    
# # Loop bar plots to html
# for metric in  tqdm(metrics):
#     fig = px.line(df_plot5_counties
#                  , x = "Year"
#                  , y = metric
#                  , color = "County Name"
#                  , title = metric
#                  , template = plot_template
#                  , markers = True
#                 )
#     fig.write_html(
#         os.path.join(
#             path_out
#             , 'plotly'
#             , 'ACS5'
#             , 'lines'
#             , ''.join([metric
#                        , '_line_'
#                        , '_ACS5_'
#                        , '_Counties_'
#                        , date.today().strftime("%Y-%m-%d")
#                        , '.html'])
#         )
#     )